# 11 - Silver: ACLED Conflict Events and Weekly Metrics

This notebook creates cleaned ACLED silver tables from the bronze event-level and weekly aggregate tables.

Outputs:

- `silver.fact_acled_events`
- `silver.fact_acled_weekly`
- `silver.fact_acled_country_year`

`silver.fact_acled_weekly` is the dashboard-ready grain for conflict intensity: one row per country, week, admin1, event type, and sub-event type. It includes normalized event and fatality rates per million residents when population is available from `silver.fact_macro_annual`.

In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

loaded_at = datetime.now(timezone.utc)

VIOLENCE_EVENT_TYPES = [
    "Battles",
    "Explosions/Remote violence",
    "Violence against civilians",
    "Riots",
]

print("Target tables:")
print("  silver.fact_acled_events")
print("  silver.fact_acled_weekly")
print("  silver.fact_acled_country_year")

In [ ]:
# Cell 2 - Input table checks
def table_exists(schema_name, table_name):
    return spark.sql(f"SHOW TABLES IN {schema_name} LIKE '{table_name}'").count() > 0


required_tables = [
    ("bronze", "acled_events_historical"),
    ("bronze", "acled_weekly_aggregated"),
    ("silver", "dim_country"),
    ("silver", "dim_bloc_membership"),
    ("silver", "fact_macro_annual"),
]

missing_tables = [f"{schema}.{table}" for schema, table in required_tables if not table_exists(schema, table)]
if missing_tables:
    raise ValueError(f"Missing required input table(s): {missing_tables}")

print("All required tables found.")

print("ACLED event input coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_events,
        COUNT(DISTINCT iso3) AS countries,
        MIN(event_date_dt) AS earliest_event_date,
        MAX(event_date_dt) AS latest_event_date,
        SUM(fatalities) AS total_fatalities
    FROM bronze.acled_events_historical
""").show()

print("ACLED weekly input coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT iso3) AS countries,
        MIN(week_start_date) AS earliest_week,
        MAX(week_start_date) AS latest_week,
        SUM(event_count) AS events,
        SUM(fatalities) AS fatalities
    FROM bronze.acled_weekly_aggregated
""").show()

In [ ]:
# Cell 3 - Shared dimensions
country_dim = spark.table("silver.dim_country").select(
    F.col("country_key"),
    F.col("country_iso3"),
    F.col("country_name"),
    F.col("region").alias("project_region"),
)

bloc_df = spark.table("silver.dim_bloc_membership").where("is_primary_analytical_bloc = true").select(
    F.col("country_iso3").alias("bloc_country_iso3"),
    F.col("bloc_code").alias("analytical_bloc_code"),
    F.col("bloc_name").alias("analytical_bloc_name"),
    F.col("valid_from").alias("bloc_valid_from"),
    F.col("valid_to").alias("bloc_valid_to"),
)

macro_population_df = spark.table("silver.fact_macro_annual").select(
    F.col("country_iso3").alias("population_country_iso3"),
    F.col("year").alias("population_year"),
    F.col("population"),
    F.col("population_millions"),
)

print("Shared dimensions loaded.")

In [ ]:
# Cell 4 - Build cleaned event-level fact table
events_raw = spark.table("bronze.acled_events_historical")

events_df = (
    events_raw.alias("e")
    .join(country_dim.alias("c"), F.col("e.iso3") == F.col("c.country_iso3"), "left")
    .withColumn("event_date", F.coalesce(F.col("e.event_date_dt"), F.to_date(F.col("e.event_date"))))
    .withColumn("week_start_date", F.expr("date_sub(event_date, pmod(dayofweek(event_date) + 5, 7))"))
    .withColumn("month_start_date", F.trunc(F.col("event_date"), "month"))
    .withColumn("year_end_date", F.expr("make_date(year, 12, 31)"))
    .join(
        bloc_df.alias("b"),
        (F.col("e.iso3") == F.col("b.bloc_country_iso3"))
        & (F.col("year_end_date") >= F.col("b.bloc_valid_from"))
        & (F.col("b.bloc_valid_to").isNull() | (F.col("year_end_date") <= F.col("b.bloc_valid_to"))),
        "left",
    )
    .select(
        F.col("e.event_id_cnty"),
        F.col("event_date"),
        F.col("week_start_date"),
        F.col("month_start_date"),
        F.col("e.year"),
        F.col("c.country_key"),
        F.col("e.iso3").alias("country_iso3"),
        F.coalesce(F.col("c.country_name"), F.col("e.project_country_name"), F.col("e.country")).alias("country_name"),
        F.col("c.project_region"),
        F.col("b.analytical_bloc_code"),
        F.col("b.analytical_bloc_name"),
        F.col("e.disorder_type"),
        F.col("e.event_type"),
        F.col("e.sub_event_type"),
        F.col("e.actor1"),
        F.col("e.assoc_actor_1"),
        F.col("e.inter1"),
        F.col("e.actor2"),
        F.col("e.assoc_actor_2"),
        F.col("e.inter2"),
        F.col("e.interaction"),
        F.col("e.civilian_targeting"),
        F.col("e.admin1"),
        F.col("e.admin2"),
        F.col("e.admin3"),
        F.col("e.location"),
        F.col("e.latitude"),
        F.col("e.longitude"),
        F.col("e.geo_precision"),
        F.col("e.source_name"),
        F.col("e.source_scale"),
        F.coalesce(F.col("e.fatalities"), F.lit(0)).cast("int").alias("fatalities"),
        F.col("e.tags"),
        F.col("e.acled_timestamp"),
        F.col("e.time_precision"),
    )
    .withColumn("is_violent_event", F.col("event_type").isin(VIOLENCE_EVENT_TYPES))
    .withColumn("is_battle", F.col("event_type") == F.lit("Battles"))
    .withColumn("is_violence_against_civilians", F.col("event_type") == F.lit("Violence against civilians"))
    .withColumn("is_explosion_remote_violence", F.col("event_type") == F.lit("Explosions/Remote violence"))
    .withColumn("is_riot_or_protest", F.col("event_type").isin(["Riots", "Protests"]))
    .withColumn("has_fatalities", F.col("fatalities") > 0)
    .withColumn("source", F.lit("acled"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

missing_country_keys = events_df.where(F.col("country_key").isNull()).select("country_iso3").distinct().count()
if missing_country_keys:
    events_df.where(F.col("country_key").isNull()).select("country_iso3").distinct().show(truncate=False)
    raise ValueError(f"ACLED events missing country dimension matches: {missing_country_keys}")

print(f"Clean event rows: {events_df.count():,}")
events_df.show(10, truncate=False)

In [ ]:
# Cell 5 - Build cleaned weekly fact table with population-normalized rates
weekly_raw = spark.table("bronze.acled_weekly_aggregated")

weekly_df = (
    weekly_raw.alias("w")
    .join(country_dim.alias("c"), F.col("w.iso3") == F.col("c.country_iso3"), "left")
    .withColumn("week_start_date", F.to_date("w.week_start_date"))
    .withColumn("week_end_date", F.date_add(F.col("week_start_date"), 6))
    .withColumn("year_end_date", F.expr("make_date(year, 12, 31)"))
    .join(
        bloc_df.alias("b"),
        (F.col("w.iso3") == F.col("b.bloc_country_iso3"))
        & (F.col("year_end_date") >= F.col("b.bloc_valid_from"))
        & (F.col("b.bloc_valid_to").isNull() | (F.col("year_end_date") <= F.col("b.bloc_valid_to"))),
        "left",
    )
    .join(
        macro_population_df.alias("p"),
        (F.col("w.iso3") == F.col("p.population_country_iso3")) & (F.col("w.year") == F.col("p.population_year")),
        "left",
    )
    .select(
        F.col("c.country_key"),
        F.col("w.iso3").alias("country_iso3"),
        F.coalesce(F.col("c.country_name"), F.col("w.project_country_name"), F.col("w.country")).alias("country_name"),
        F.col("c.project_region"),
        F.col("b.analytical_bloc_code"),
        F.col("b.analytical_bloc_name"),
        F.col("w.year"),
        F.col("week_start_date"),
        F.col("week_end_date"),
        F.col("w.admin1"),
        F.col("w.event_type"),
        F.col("w.sub_event_type"),
        F.col("w.event_count").cast("long").alias("event_count"),
        F.coalesce(F.col("w.fatalities"), F.lit(0)).cast("long").alias("fatalities"),
        F.col("w.distinct_events").cast("long").alias("distinct_events"),
        F.col("p.population"),
        F.col("p.population_millions"),
    )
    .withColumn("is_violent_event", F.col("event_type").isin(VIOLENCE_EVENT_TYPES))
    .withColumn("event_count_per_million", F.when(F.col("population") > 0, F.col("event_count") / F.col("population") * 1_000_000.0))
    .withColumn("fatalities_per_million", F.when(F.col("population") > 0, F.col("fatalities") / F.col("population") * 1_000_000.0))
    .withColumn("source", F.lit("acled"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

missing_weekly_country_keys = weekly_df.where(F.col("country_key").isNull()).select("country_iso3").distinct().count()
if missing_weekly_country_keys:
    weekly_df.where(F.col("country_key").isNull()).select("country_iso3").distinct().show(truncate=False)
    raise ValueError(f"ACLED weekly rows missing country dimension matches: {missing_weekly_country_keys}")

print(f"Weekly rows: {weekly_df.count():,}")
weekly_df.show(10, truncate=False)

In [ ]:
# Cell 6 - Build country-year ACLED summary fact table
country_year_df = (
    weekly_df
    .groupBy("country_key", "country_iso3", "country_name", "project_region", "analytical_bloc_code", "analytical_bloc_name", "year")
    .agg(
        F.sum("event_count").alias("total_events"),
        F.sum(F.when(F.col("is_violent_event"), F.col("event_count")).otherwise(F.lit(0))).alias("violent_events"),
        F.sum(F.when(F.col("event_type") == "Battles", F.col("event_count")).otherwise(F.lit(0))).alias("battle_events"),
        F.sum(F.when(F.col("event_type") == "Violence against civilians", F.col("event_count")).otherwise(F.lit(0))).alias("violence_against_civilians_events"),
        F.sum(F.when(F.col("event_type") == "Explosions/Remote violence", F.col("event_count")).otherwise(F.lit(0))).alias("explosions_remote_violence_events"),
        F.sum(F.when(F.col("event_type") == "Protests", F.col("event_count")).otherwise(F.lit(0))).alias("protest_events"),
        F.sum(F.when(F.col("event_type") == "Riots", F.col("event_count")).otherwise(F.lit(0))).alias("riot_events"),
        F.sum("fatalities").alias("fatalities"),
        F.countDistinct("admin1").alias("admin1_areas_with_events"),
        F.max("population").alias("population"),
        F.max("population_millions").alias("population_millions"),
    )
    .withColumn("events_per_million", F.when(F.col("population") > 0, F.col("total_events") / F.col("population") * 1_000_000.0))
    .withColumn("violent_events_per_million", F.when(F.col("population") > 0, F.col("violent_events") / F.col("population") * 1_000_000.0))
    .withColumn("fatalities_per_million", F.when(F.col("population") > 0, F.col("fatalities") / F.col("population") * 1_000_000.0))
    .withColumn("source", F.lit("acled"))
    .withColumn("loaded_at", F.lit(loaded_at))
)

print(f"Country-year rows: {country_year_df.count():,}")
country_year_df.show(10, truncate=False)

In [ ]:
# Cell 7 - Write silver ACLED tables
spark.sql("DROP TABLE IF EXISTS silver.fact_acled_events")
spark.sql("DROP TABLE IF EXISTS silver.fact_acled_weekly")
spark.sql("DROP TABLE IF EXISTS silver.fact_acled_country_year")

(events_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fact_acled_events"))

(weekly_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fact_acled_weekly"))

(country_year_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.fact_acled_country_year"))

print("Write complete:")
print("  silver.fact_acled_events")
print("  silver.fact_acled_weekly")
print("  silver.fact_acled_country_year")

In [ ]:
# Cell 8 - Verification and sanity checks
print("Event-level coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_events,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(event_date) AS earliest_event_date,
        MAX(event_date) AS latest_event_date,
        SUM(fatalities) AS total_fatalities,
        SUM(CASE WHEN is_violent_event THEN 1 ELSE 0 END) AS violent_events
    FROM silver.fact_acled_events
""").show()

print("Weekly coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS weekly_rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(week_start_date) AS earliest_week,
        MAX(week_start_date) AS latest_week,
        SUM(event_count) AS total_events_reconstructed,
        SUM(fatalities) AS total_fatalities_reconstructed
    FROM silver.fact_acled_weekly
""").show()

print("Latest country-year conflict intensity:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        country_iso3,
        country_name,
        year,
        total_events,
        violent_events,
        fatalities,
        ROUND(events_per_million, 1) AS events_per_million,
        ROUND(fatalities_per_million, 1) AS fatalities_per_million
    FROM silver.fact_acled_country_year
    WHERE year = (SELECT MAX(year) FROM silver.fact_acled_country_year)
    ORDER BY violent_events DESC
""").show(25, truncate=False)

print("Top event types since 2010:")
spark.sql("""
    SELECT
        event_type,
        COUNT(*) AS events,
        SUM(fatalities) AS fatalities
    FROM silver.fact_acled_events
    GROUP BY event_type
    ORDER BY events DESC
""").show(truncate=False)

print("Cameroon admin1 conflict intensity, latest 3 years:")
spark.sql("""
    SELECT
        admin1,
        SUM(event_count) AS events,
        SUM(fatalities) AS fatalities,
        ROUND(SUM(event_count_per_million), 1) AS summed_weekly_events_per_million
    FROM silver.fact_acled_weekly
    WHERE country_iso3 = 'CMR'
      AND year >= (SELECT MAX(year) FROM silver.fact_acled_weekly) - 2
    GROUP BY admin1
    ORDER BY events DESC
    LIMIT 10
""").show(truncate=False)